# Harmonic Oscillator Advantage: n=20 Confirmatory Rerun

**Purpose:** Experiment 19's headline Harmonic result (Panda advantage +0.370,
p=0.004) rests on n_windows=8, the same evidentiary tier the sensor-heterogeneity
finding sat at before it collapsed at n=20 (Experiments 33-34). This reruns it
at n=20 using the real Panda-vs-Chronos advantage metric (not a self-referential
ablation), with no methodology changes -- same model loading, same evaluate()
harness, same Harmonic generator already validated in this project's own OOD
campaign.

**Provenance:** model loading + evaluate()/panda_forecast/chronos_forecast are
verbatim from `fixed_experiments.ipynb` (Cells 0-1). The Harmonic loader
(`simulate_harmonic`, `load_harmonic`) is verbatim from `eval-nb.ipynb`'s OOD
loaders section, the same loader already used successfully for real evaluation
in that notebook's 100k OOD campaign -- not a new or modified generator.

**Pre-registered comparison target:** Experiment 19's original result,
H=96, n_windows=8, Panda advantage=+0.370, p=0.004. This rerun uses n_windows=20
at H=96 (primary) and additionally reports H=192/336 (bonus, not required for
the confirmatory question, included since the harness supports it trivially).


In [1]:
# ============================================================
# CELL 1 -- IMPORTS + MODEL LOADING (published checkpoints)
# Verbatim from fixed_experiments.ipynb, Cell 0.
# ============================================================
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

import sys
sys.path.insert(0, './panda')  # local run: panda repo cloned in the working directory

from panda.patchtst.pipeline import PatchTSTPipeline
from chronos import ChronosPipeline

panda_model = PatchTSTPipeline.from_pretrained(
    mode="predict",
    pretrain_path="GilpinLab/panda",
    device_map=device,
)

chronos_model = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.bfloat16,
)

print("Models loaded.")


Device: cpu
Models loaded.


In [2]:
# ============================================================
# CELL 2 -- EVALUATION HARNESS
# Verbatim from fixed_experiments.ipynb, Cell 1 (mae, instance_norm_window,
# CONTEXT_LEN, panda_forecast, chronos_forecast, evaluate). No changes.
# ============================================================
def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred)**2))

def instance_norm_window(x_CT):
    """x_CT: (C, T). Normalise per channel using this window only."""
    mu  = x_CT.mean(axis=1, keepdims=True)
    std = x_CT.std( axis=1, keepdims=True) + 1e-8
    return (x_CT - mu) / std, mu, std

CONTEXT_LEN = 512

def panda_forecast(context_np, horizon):
    """context_np: (C, T) normalised. Returns (C, horizon)."""
    TRAIN_H   = 128
    remaining = horizon
    ctx       = context_np.copy()
    preds     = []
    while remaining > 0:
        h         = min(TRAIN_H, remaining)
        context_t = torch.tensor(ctx.T, dtype=torch.float32)
        with torch.no_grad():
            pred = panda_model.predict(
                context_t, h,
                limit_prediction_length=False,
                sliding_context=True,
            )
        p = pred.squeeze().cpu().numpy()
        if p.ndim == 1:
            p = p[:, None]
        if p.shape[0] != context_np.shape[0]:
            p = p.T
        preds.append(p[:, :h])
        ctx       = np.concatenate([ctx[:, h:], p[:, :h]], axis=1)
        remaining -= h
    return np.concatenate(preds, axis=1)  # (C, horizon)

def chronos_forecast(context_np, horizon):
    """Batched -- all channels in one call."""
    ctx = torch.tensor(context_np, dtype=torch.float32)
    with torch.no_grad():
        out = chronos_model.predict(
            ctx, prediction_length=horizon, num_samples=1
        )
    return out[:, 0, :].cpu().numpy()  # (C, horizon)

def evaluate(data_CT, horizon, n_windows=8, label="",
             fn_a=None, fn_b=None,
             name_a="panda", name_b="chronos"):
    """
    data_CT: (C, T) RAW. Normalises each window independently.
    fn_a, fn_b: forecast functions (context_normed, horizon) -> (C, H)
    """
    if fn_a is None: fn_a = panda_forecast
    if fn_b is None: fn_b = chronos_forecast

    C, T      = data_CT.shape
    max_start = T - CONTEXT_LEN - horizon
    if max_start <= 0:
        print(f"  [SKIP] {label}: T={T} too short")
        return None

    starts = np.linspace(0, max_start, n_windows, dtype=int)
    mae_a, mae_b = [], []

    for s in starts:
        ctx_raw = data_CT[:, s : s + CONTEXT_LEN]
        tgt_raw = data_CT[:, s + CONTEXT_LEN : s + CONTEXT_LEN + horizon]
        ctx_norm, mu, std = instance_norm_window(ctx_raw)
        tgt_norm = (tgt_raw - mu) / std

        mae_a.append(mae(tgt_norm, fn_a(ctx_norm, horizon)))
        mae_b.append(mae(tgt_norm, fn_b(ctx_norm, horizon)))

    diff = np.array(mae_b) - np.array(mae_a)
    try:
        if np.any(diff != 0):
            _, pval = wilcoxon(diff, alternative="greater")
        else:
            pval = 1.0
    except Exception:
        pval = np.nan

    adv = np.median(mae_b) - np.median(mae_a)
    sig = " *" if pval < 0.05 else (" ~" if pval < 0.10 else "")

    result = {
        "label"         : label,
        "horizon"       : horizon,
        "name_a"        : name_a,
        "name_b"        : name_b,
        f"{name_a}_mae" : np.median(mae_a),
        f"{name_a}_iqr" : np.percentile(mae_a,75)-np.percentile(mae_a,25),
        f"{name_b}_mae" : np.median(mae_b),
        f"{name_b}_iqr" : np.percentile(mae_b,75)-np.percentile(mae_b,25),
        "advantage_mae" : adv,
        "wilcoxon_p"    : pval,
    }

    print(
        f"  {label:24s}  H={horizon:4d}  "
        f"{name_a}={np.median(mae_a):.4f}[+/-{result[f'{name_a}_iqr']:.4f}]  "
        f"{name_b}={np.median(mae_b):.4f}[+/-{result[f'{name_b}_iqr']:.4f}]  "
        f"Adv={adv:+.4f}  p={pval:.4f}{sig}"
    )
    return result

print("Helpers defined.")


Helpers defined.


In [3]:
# ============================================================
# CELL 3 -- HARMONIC LOADER
# Verbatim from eval-nb.ipynb's OOD Loaders section (the load_harmonic()
# already used for real evaluation in that notebook's OOD campaign) --
# n_steps=4000, omega=1.0, seed=42, skip first 500 steps as transient.
# ============================================================
SEED = 42

def simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED):
    rng = np.random.default_rng(seed)
    dt  = 0.05
    x, v = float(rng.standard_normal()), float(rng.standard_normal())
    traj = []
    for _ in range(n_steps):
        traj.append(x)
        x_new = x + v * dt
        v_new = v - omega**2 * x * dt
        x, v  = x_new, v_new
    return np.array(traj, dtype=np.float32)

def load_harmonic():
    series = simulate_harmonic(n_steps=4000, omega=1.0, seed=SEED)
    return series[500:][None, :]  # (1, 3500)

data_harmonic = load_harmonic()
print(f"Harmonic trajectory loaded: shape={data_harmonic.shape}")


Harmonic trajectory loaded: shape=(1, 3500)


In [4]:
# ============================================================
# CELL 4 -- RUN: n_windows=20, H=96 (primary), H=192/336 (bonus)
# ============================================================
N_WINDOWS = 20
HORIZONS = [96, 192, 336]  # 96 is the pre-registered comparison target

harmonic_results = []
print("Harmonic oscillator, n_windows=20:")
print()
for H in HORIZONS:
    r = evaluate(data_harmonic, H, n_windows=N_WINDOWS,
                 label=f"Harmonic_H{H}_n20",
                 name_a="panda", name_b="chronos")
    if r:
        harmonic_results.append(r)

df_harmonic_n20 = pd.DataFrame(harmonic_results)
df_harmonic_n20.to_csv("harmonic_n20_confirmatory_results.csv", index=False)
print()
print("Saved harmonic_n20_confirmatory_results.csv")


Harmonic oscillator, n_windows=20:



We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H96_n20          H=  96  panda=0.0691[+/-0.0339]  chronos=0.3909[+/-0.1857]  Adv=+0.3218  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H192_n20         H= 192  panda=0.1352[+/-0.0615]  chronos=0.7572[+/-0.6237]  Adv=+0.6220  p=0.0000 *


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

  Harmonic_H336_n20         H= 336  panda=0.3297[+/-0.1659]  chronos=1.4012[+/-0.5368]  Adv=+1.0715  p=0.0000 *

Saved harmonic_n20_confirmatory_results.csv


In [5]:
# ============================================================
# CELL 5 -- COMPARISON AGAINST THE ORIGINAL EXPERIMENT 19 RESULT
# Original (log, Section 6, Experiment 19): H=96, n_windows=8,
# Panda advantage = +0.370, p = 0.004.
# ============================================================
ORIGINAL_ADV = 0.370
ORIGINAL_P = 0.004
ORIGINAL_N = 8

h96_row = df_harmonic_n20[df_harmonic_n20["horizon"] == 96].iloc[0]
new_adv = h96_row["advantage_mae"]
new_p = h96_row["wilcoxon_p"]

print("Comparison at H=96 (the pre-registered target):")
print(f"  Original (n={ORIGINAL_N}):  advantage = {ORIGINAL_ADV:+.3f}, p = {ORIGINAL_P:.4f}")
print(f"  This run (n={N_WINDOWS}): advantage = {new_adv:+.3f}, p = {new_p:.4f}")
print()

ratio = new_adv / ORIGINAL_ADV if ORIGINAL_ADV != 0 else float('nan')
print(f"  Effect size ratio (new/original): {ratio:.2f}x")

if new_p < 0.05 and abs(ratio - 1) < 0.5:
    print("  -> CONFIRMED: significant at n=20, effect size in the same rough range as n=8.")
elif new_p < 0.05:
    print("  -> PARTIALLY CONFIRMED: still significant at n=20, but effect size has "
          + "shifted substantially from the original (ratio outside 0.5-1.5x) -- "
          + "worth noting the magnitude, not just the direction, changed.")
else:
    print("  -> NOT CONFIRMED at n=20: does not reach significance. Per this project's "
          + "own precedent (the heterogeneity bottleneck), an n=8 result not surviving "
          + "n=20 is a real, previously-observed pattern in this investigation, not "
          + "an unprecedented one.")

print()
print("Full n=20 results (all horizons):")
print(df_harmonic_n20[["label", "horizon", "panda_mae", "chronos_mae", "advantage_mae", "wilcoxon_p"]]
      .round(4).to_string(index=False))


Comparison at H=96 (the pre-registered target):
  Original (n=8):  advantage = +0.370, p = 0.0040
  This run (n=20): advantage = +0.322, p = 0.0000

  Effect size ratio (new/original): 0.87x
  -> CONFIRMED: significant at n=20, effect size in the same rough range as n=8.

Full n=20 results (all horizons):
            label  horizon  panda_mae  chronos_mae  advantage_mae  wilcoxon_p
 Harmonic_H96_n20       96     0.0691       0.3909         0.3218         0.0
Harmonic_H192_n20      192     0.1352       0.7572         0.6220         0.0
Harmonic_H336_n20      336     0.3297       1.4012         1.0715         0.0
